# LDA e QDA em dry_bean

## Imports e visualizações

Primeiro, vamos começar pelas importações necessárias para a biblioteca e a aplicação no dataset, fazendo algumas visualizações de dados das features em dry_bean.

In [1]:
import sys
import zipfile
from pathlib import Path

# Diretórios candidatos: instalação local e pasta atual.
BASE_CANDIDATES = [
    Path("/content/drive/MyDrive/ML-Project"),
    Path.home() / "Downloads" / "ML-Project",
    Path.cwd(),
    Path("/content"),
    Path("/mnt/data"),
]

RBG_SRC = None
BASE = None
for base_candidate in dict.fromkeys(BASE_CANDIDATES):
    # Primeiro, verificamos se a biblioteca já está extraída.
    for rbg_folder in (base_candidate / "rbg", base_candidate / "rbg-main", base_candidate):
        src_candidate = rbg_folder / "src"
        if (src_candidate / "rbg").is_dir():
            BASE = base_candidate
            RBG_SRC = src_candidate
            break
    if RBG_SRC is not None:
        break

    # Caso a biblioteca esteja somente no arquivo ZIP, extraímos na pasta candidata.
    zip_candidates = (base_candidate / "rbg-main.zip", base_candidate / "rbg.zip")
    for zip_candidate in zip_candidates:
        if zip_candidate.is_file():
            with zipfile.ZipFile(zip_candidate, "r") as arquivo_zip:
                arquivo_zip.extractall(base_candidate)
            for rbg_folder in (base_candidate / "rbg", base_candidate / "rbg-main"):
                src_candidate = rbg_folder / "src"
                if (src_candidate / "rbg").is_dir():
                    BASE = base_candidate
                    RBG_SRC = src_candidate
                    break
            if RBG_SRC is not None:
                break
    if RBG_SRC is not None:
        break

if RBG_SRC is None:
    raise FileNotFoundError(
        "Não encontrei src/rbg nem uma biblioteca rbg-main.zip. "
        "Coloque o ZIP em uma das pastas candidatas ou ajuste BASE_CANDIDATES."
    )

if str(RBG_SRC) not in sys.path:
    sys.path.insert(0, str(RBG_SRC))

print("RBG encontrada:", RBG_SRC)
print("Python em uso:", sys.executable)


RBG encontrada: C:\Users\User\Downloads\ML-Project\rbg\src
Python em uso: c:\Users\User\AppData\Local\Programs\Python\Python313\python.exe


In [2]:
import altair as alt
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from rbg.ModelSelection.StratifiedKFold import StratifiedKFold
from rbg.Preprocessing.Standard import Standard
from rbg.Classification.LDA import LDA
from rbg.Classification.QDA import QDA
from rbg.Utilities.ClassificationMetrics import (
    Accuracy, Precision, Recall, F1Score
)
from rbg.Unsupervised.PCA import PCA


In [3]:

from IPython.display import display


alt.data_transformers.disable_max_rows()

CLASS_COLORS = {
    "DERMASON": "#0173B2",
    "SIRA":     "#DE8F05",
    "SEKER":    "#029E73",
    "HOROZ":    "#D55E00",
    "CALI":     "#CC78BC",
    "BARBUNYA": "#CA9161",
    "BOMBAY":   "#FBAFE4",
}

def stylize(chart):
    return (
        chart
        .configure_axis(labelFontSize=10, titleFontSize=11, gridColor="#E5E5E5")
        .configure_legend(symbolOpacity=1, labelFontSize=10, titleFontSize=11)
        .configure_title(fontSize=14)
    )

def macro_f1_score(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    """Calcula o F1-macro usando os valores por classe retornados pela RBG."""
    return float(np.mean(F1Score(np.asarray(y_pred), np.asarray(y_true))))

In [4]:
dry_bean = fetch_ucirepo(id=602)
df = pd.concat([dry_bean.data.features, dry_bean.data.targets], axis=1)

df = df.rename(columns={"AspectRation": "AspectRatio", "roundness": "Roundness"})

TARGET = "Class"

FEATURES = [
    "Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRatio", "Eccentricity", "ConvexArea", "EquivDiameter",
    "Extent", "Solidity", "Roundness", "Compactness",
    "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4"
]

display(df[:5])
display(df["Class"].value_counts())


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64

## Processamento dos dados

O conjunto de dados Dry Bean contém inicialmente 13.611 observações, 16 variáveis numéricas e uma variável-alvo correspondente à variedade do feijão. A variável de classificação apresenta sete categorias: BARBUNYA, BOMBAY, CALI, DERMASON, HOROZ, SEKER e SIRA.

Antes de iniciar os experimentos, removemos as duplicatas exatas identificadas na análise exploratória. Foram encontradas 68 linhas repetidas, resultando em uma base com 13.543 observações. Mantemos o conjunto original em `df` e utilizamos a cópia sem duplicatas, `df_model`, nas etapas de treinamento e avaliação.

Em seguida, realizamos uma divisão estratificada entre treinamento e teste, utilizando a primeira partição de um `StratifiedKFold` com cinco folds, embaralhamento e semente aleatória igual a 42. Essa divisão resulta em 10.831 observações para treinamento e 2.712 para o teste externo, correspondendo aproximadamente a 80% e 20% dos dados, respectivamente.

A estratificação busca preservar a distribuição das sete classes nos dois subconjuntos. As proporções observadas no treinamento e no teste são bastante próximas, o que permite realizar comparações em condições mais equilibradas.

As versões brutas das variáveis explicativas, armazenadas em `X_train_raw` e `X_test_raw`, são preservadas para que o pré-processamento possa ser ajustado corretamente em cada etapa. A padronização é realizada utilizando a implementação `Standard` da biblioteca RBG, enquanto o PCA é empregado nas configurações que investigam a redução dimensional.

A validação cruzada é realizada exclusivamente sobre o conjunto de treinamento externo. Serão comparadas oito configurações: LDA e QDA sem PCA e ambos os classificadores com limiares de variância explicada de 90%, 95% e 99%.

Para evitar vazamento de informações, o `Standard` e, quando necessário, o PCA são ajustados novamente utilizando apenas as observações de treinamento de cada fold. As observações de validação recebem as transformações previamente ajustadas, sem participar da estimação de seus parâmetros.

In [5]:

# Remoção das duplicatas exatas identificadas na EDA antes da separação.
# A análise exploratória encontrou 68 linhas duplicadas; mantemos o df original dos gráficos, mas treinamos os classificadores em uma cópia sem repetições.

df_model = df.drop_duplicates().reset_index(drop=True)

print(f"Observações originais: {len(df)}")
print(f"Duplicatas exatas removidas: {len(df) - len(df_model)}")
print(f"Observações após a remoção: {len(df_model)}")

# Separação estratificada entre treino e teste (aproximadamente 80/20).
# Um fold de StratifiedKFold é reservado como teste externo.

X = df_model[FEATURES].to_numpy(dtype=float)
y = df_model[TARGET].to_numpy()

holdout_splitter = StratifiedKFold(k=5, shuffle=True, random_state=42)
idx_treino, idx_teste = next(holdout_splitter.split(X, y))

# Preservamos os dados brutos para reajustar Standard/PCA dentro de cada fold.

X_train_raw = X[idx_treino].copy()
X_test_raw = X[idx_teste].copy()

y_train = y[idx_treino].copy()
y_test = y[idx_teste].copy()

# Limiar de variância utilizado nos experimentos principais com PCA.
PCA_VARIANCE_TARGET = 0.90

# Classes utilizadas pelos classificadores LDA e QDA da RBG.
CLASSES_LDA_QDA = np.unique(y_train)

print("Classes:", CLASSES_LDA_QDA)

print(f"Treino: {X_train_raw.shape[0]} observações | Teste: {X_test_raw.shape[0]} observações")

print("Proporções no treino:")
display(
    pd.Series(y_train).value_counts(normalize=True)
    .sort_index().rename("proporção")
)

print("Proporções no teste:")
display(
    pd.Series(y_test).value_counts(normalize=True)
    .sort_index().rename("proporção")
)


Observações originais: 13611
Duplicatas exatas removidas: 68
Observações após a remoção: 13543
Classes: ['BARBUNYA' 'BOMBAY' 'CALI' 'DERMASON' 'HOROZ' 'SEKER' 'SIRA']
Treino: 10831 observações | Teste: 2712 observações
Proporções no treino:


BARBUNYA    0.097590
BOMBAY      0.038501
CALI        0.120395
DERMASON    0.261841
HOROZ       0.137383
SEKER       0.149663
SIRA        0.194627
Name: proporção, dtype: float64

Proporções no teste:


BARBUNYA    0.097714
BOMBAY      0.038717
CALI        0.120206
DERMASON    0.261799
HOROZ       0.137168
SEKER       0.149705
SIRA        0.194690
Name: proporção, dtype: float64

A implementação `Standard` estima, durante o ajuste (`fit`), a média e o desvio-padrão de cada variável, utilizando o cálculo de desvio-padrão com `ddof=0`. Posteriormente, o método `transform` aplica esses parâmetros às observações recebidas.

Na célula seguinte, ajustamos o padronizador exclusivamente com os dados do treinamento externo e aplicamos a transformação também ao conjunto de teste. A média aproximadamente nula e o desvio-padrão aproximadamente igual a um nas variáveis padronizadas confirmam o comportamento esperado da transformação.

Essa etapa também funciona como uma verificação da implementação. Nos experimentos propriamente ditos, o `Standard` é reajustado dentro de cada fold da validação cruzada. Na avaliação externa, é ajustado novamente utilizando todo o treinamento externo e aplicado ao teste. Dessa forma, a padronização utilizada nos experimentos respeita a separação entre treinamento e validação.

In [6]:
standard_final = Standard().fit(X_train_raw)
X_train_std = standard_final.transform(X_train_raw)
X_test_std = standard_final.transform(X_test_raw)

print("Média aproximada das features padronizadas no treino:")
print(np.round(X_train_std.mean(axis=0), 6))
print("Desvio-padrão aproximado das features padronizadas no treino:")
print(np.round(X_train_std.std(axis=0), 6))


Média aproximada das features padronizadas no treino:
[-0.  0.  0. -0.  0.  0. -0. -0. -0. -0.  0. -0.  0.  0. -0. -0.]
Desvio-padrão aproximado das features padronizadas no treino:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## Métricas

As funções da RBG recebem os argumentos na ordem (y_pred, y_true). Accuracy retorna um escalar; Precision, Recall e F1Score retornam vetores por classe, ordenados pelas classes presentes na união de y_pred e y_true. Calculamos as médias macro explicitamente. O F1 ponderado usa o suporte de cada classe. Como a base é desbalanceada, o F1-macro será a métrica principal de comparação.

In [7]:
def preparar_rotulos(y_pred: np.ndarray, y_true: np.ndarray):
    """Padroniza e valida os vetores de rótulos."""

    y_pred = np.asarray(y_pred).ravel()
    y_true = np.asarray(y_true).ravel()

    if y_pred.shape != y_true.shape:
        raise ValueError("y_pred e y_true precisam ter o mesmo tamanho.")

    if y_true.size == 0:
        raise ValueError("Os vetores de rótulos não podem estar vazios.")

    classes_metricas = np.unique(np.concatenate([y_true, y_pred]))

    return y_pred, y_true, classes_metricas


def metricas_classificacao(
    y_pred: np.ndarray,
    y_true: np.ndarray
) -> dict[str, float]:
    """Calcula métricas globais utilizando as funções da RBG."""

    y_pred, y_true, classes_metricas = preparar_rotulos(
        y_pred, y_true
    )

    precisao_por_classe = Precision(y_pred, y_true)
    recall_por_classe = Recall(y_pred, y_true)
    f1_por_classe = F1Score(y_pred, y_true)

    # Confere se os vetores retornados seguem a ordem esperada das classes.
    if not all(
        metricas.shape == classes_metricas.shape
        for metricas in (
            precisao_por_classe,
            recall_por_classe,
            f1_por_classe
        )
    ):
        raise ValueError("Dimensões inesperadas nas métricas retornadas pela RBG.")

    # Suporte: número de observações reais pertencentes a cada classe.
    suporte = np.array([
        np.sum(y_true == classe)
        for classe in classes_metricas
    ], dtype=float)

    return {
        "Acurácia": float(Accuracy(y_pred, y_true)),
        "Precisão-macro": float(np.mean(precisao_por_classe)),
        "Recall-macro": float(np.mean(recall_por_classe)),
        "F1-macro": float(np.mean(f1_por_classe)),
        "F1-ponderado": float(
            np.average(f1_por_classe, weights=suporte)
        )
    }


def tabela_metricas_classe(
    y_pred: np.ndarray,
    y_true: np.ndarray
) -> pd.DataFrame:
    """Retorna precisão, recall, F1 e suporte para cada classe."""

    y_pred, y_true, classes_metricas = preparar_rotulos(
        y_pred, y_true
    )

    precisao = Precision(y_pred, y_true)
    recall = Recall(y_pred, y_true)
    f1 = F1Score(y_pred, y_true)

    return pd.DataFrame({
        "Precisão": precisao,
        "Recall": recall,
        "F1-score": f1,
        "Suporte": [
            np.sum(y_true == classe)
            for classe in classes_metricas
        ]
    }, index=pd.Index(classes_metricas, name="Classe"))


def matriz_confusao_rbg(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    classes: np.ndarray
) -> pd.DataFrame:
    """
    Monta a matriz de confusão.
    Linhas representam as classes reais e colunas, as previstas.
    Atenção: aqui a ordem dos argumentos é (y_true, y_pred).
    """

    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    classes = np.asarray(classes)

    if y_true.shape != y_pred.shape:
        raise ValueError("y_true e y_pred precisam ter o mesmo tamanho.")

    if y_true.size == 0:
        raise ValueError("Os vetores de rótulos não podem estar vazios.")

    if classes.ndim != 1 or classes.size == 0:
        raise ValueError("A lista de classes precisa ser um vetor não vazio.")

    if np.unique(classes).size != classes.size:
        raise ValueError("A lista de classes não pode conter duplicatas.")

    indice = {classe: i for i, classe in enumerate(classes)}
    matriz = np.zeros((len(classes), len(classes)), dtype=int)

    for verdadeiro, previsto in zip(y_true, y_pred):

        if verdadeiro not in indice or previsto not in indice:
            raise ValueError(
                "Foi encontrada uma classe fora da lista informada."
            )

        matriz[indice[verdadeiro], indice[previsto]] += 1

    return pd.DataFrame(
        matriz,
        index=pd.Index(classes, name="Classe real"),
        columns=pd.Index(classes, name="Classe prevista")
    )


## Diagnóstico das matrizes de covariância

Antes da realização dos experimentos de classificação, investigamos as propriedades numéricas das matrizes de covariância utilizadas pelos classificadores LDA e QDA da biblioteca RBG.

A análise considera dois espaços de representação: o espaço original, composto pelas 16 variáveis padronizadas, e o espaço reduzido pelo PCA, utilizando o limiar de variância explicada de 90%.

Essa investigação é importante porque o LDA utiliza uma matriz de covariância agrupada (*pooled covariance*), enquanto o QDA estima uma matriz de covariância específica para cada classe. A inversão dessas matrizes é uma operação relevante para o funcionamento dos classificadores, de modo que sua estrutura e seu condicionamento numérico merecem atenção.

No espaço original padronizado, todas as matrizes apresentaram posto completo, igual à sua dimensão. Apesar disso, os números de condição foram elevados: aproximadamente 8,32 × 10⁵ para a matriz agrupada do LDA e valores entre 9,50 × 10⁶ e 1,26 × 10⁸ para as matrizes específicas do QDA. Esses valores indicam que, embora as matrizes sejam inversíveis no diagnóstico realizado, algumas apresentam condicionamento numérico desfavorável.

Após a aplicação do PCA, a representação passou a utilizar quatro componentes, responsáveis por reter aproximadamente 94,98% da variância. Nesse espaço, todas as matrizes continuaram apresentando posto completo, enquanto seus números de condição passaram a variar aproximadamente entre 2,60 e 11,97.

Portanto, os resultados indicam que a redução dimensional proporcionada pelo PCA melhora significativamente o condicionamento numérico das matrizes de covariância. Esse comportamento justifica investigar seu efeito nos classificadores, embora uma melhora na estabilidade numérica não implique, necessariamente, maior desempenho preditivo.

Ressaltamos que este diagnóstico possui caráter exploratório e não substitui o ajuste realizado dentro de cada fold da validação cruzada. Nos experimentos, o pré-processamento é novamente ajustado apenas com os dados de treinamento de cada partição.

In [8]:
def diagnosticar_covariancias(
    Z: np.ndarray, y: np.ndarray, descricao: str
) -> list[dict[str, object]]:
    classes = np.unique(y)
    n, p = Z.shape

    residuos = np.vstack([
        Z[y == classe] - Z[y == classe].mean(axis=0)
        for classe in classes
    ])
    cov_lda = (residuos.T @ residuos) / (n - len(classes))

    matrizes = [("LDA — pooled", cov_lda)]
    for classe in classes:
        Z_classe = Z[y == classe]
        centrado = Z_classe - Z_classe.mean(axis=0)
        cov_qda = (centrado.T @ centrado) / (len(Z_classe) - 1)
        matrizes.append((f"QDA — {classe}", cov_qda))

    linhas = []
    for nome, cov in matrizes:
        linhas.append({
            "Espaço": descricao,
            "Matriz": nome,
            "Dimensão": p,
            "Posto": int(np.linalg.matrix_rank(cov)),
            "Número de condição": float(np.linalg.cond(cov)),
        })
    return linhas


scaler_diag = Standard().fit(X_train_raw)
X_train_diag_std = scaler_diag.transform(X_train_raw)
pca_diag = PCA(n_components=PCA_VARIANCE_TARGET).fit(X_train_diag_std)
X_train_diag_pca = pca_diag.transform(X_train_diag_std)

linhas_diagnostico = (
    diagnosticar_covariancias(
        X_train_diag_std, y_train, "16 features padronizadas"
    )
    + diagnosticar_covariancias(
        X_train_diag_pca, y_train, "PCA 90%"
    )
)

df_diagnostico_cov = pd.DataFrame(linhas_diagnostico)
display(df_diagnostico_cov.round(3))
print(
    f"PCA reteve {pca_diag.components.shape[0]} componentes e "
    f"{np.sum(pca_diag.explained_variance_ratio) * 100:.2f}% da variância."
)

,Espaço,Matriz,Dimensão,Posto,Número de condição
0,16 features padronizadas,LDA — pooled,16,16,8.315105e+05
1,16 features padronizadas,QDA — BARBUNYA,16,16,9.499585e+06
2,16 features padronizadas,QDA — BOMBAY,16,16,2.225963e+07
3,16 features padronizadas,QDA — CALI,16,16,3.239341e+07
4,16 features padronizadas,QDA — DERMASON,16,16,2.649901e+07
5,16 features padronizadas,QDA — HOROZ,16,16,2.900032e+07
6,16 features padronizadas,QDA — SEKER,16,16,1.255115e+08
7,16 features padronizadas,QDA — SIRA,16,16,5.261419e+07
8,PCA 90%,LDA — pooled,4,4,2.597000e+00
9,PCA 90%,QDA — BARBUNYA,4,4,2.807000e+00


PCA reteve 4 componentes e 94.98% da variância.


## Validação cruzada estratificada

A comparação dos classificadores é realizada por meio de validação cruzada estratificada com cinco folds, utilizando exclusivamente o conjunto de treinamento externo. Todas as configurações utilizam exatamente as mesmas partições, permitindo uma comparação consistente entre os resultados.

São investigadas oito configurações: LDA e QDA sem redução dimensional e os mesmos classificadores com PCA, considerando limiares de variância explicada de 90%, 95% e 99%.

Em cada fold, o `Standard` é ajustado apenas com as observações pertencentes ao subconjunto de treinamento. Quando há redução dimensional, o PCA também é ajustado exclusivamente nesse subconjunto. Em seguida, as transformações são aplicadas às observações de validação, preservando a separação necessária para uma avaliação adequada.

A principal métrica de comparação é o F1-macro médio, que atribui o mesmo peso ao desempenho de cada classe, independentemente de sua frequência. Também são consideradas a acurácia, a precisão-macro, o recall-macro e o F1 ponderado. O desvio-padrão do F1-macro entre os folds é utilizado como medida descritiva da variação do desempenho entre as partições.

É importante observar que os percentuais associados ao PCA representam limiares mínimos de variância explicada. Como o número de componentes é inteiro, a variância efetivamente retida pode ultrapassar o valor solicitado.

In [9]:
N_FOLDS_LDA_QDA = 5

cv_lda_qda = StratifiedKFold(
    k=N_FOLDS_LDA_QDA,
    shuffle=True,
    random_state=42
)

# Guardamos os folds para garantir que todos os experimentos utilizem
# exatamente as mesmas divisões de treino e validação.
folds_cv = list(cv_lda_qda.split(X_train_raw, y_train))


def executar_experimento_cv(
    nome_modelo: str,
    classe_modelo,
    usar_pca: bool
) -> pd.DataFrame:

    resultados = []

    for numero_fold, (idx_tr, idx_val) in enumerate(folds_cv, start=1):

        X_fold_tr = X_train_raw[idx_tr]
        X_fold_val = X_train_raw[idx_val]

        y_fold_tr = y_train[idx_tr]
        y_fold_val = y_train[idx_val]

        # 1. Padronização: ajustada somente no fold de treino.
        scaler_fold = Standard().fit(X_fold_tr)

        X_fold_tr_std = scaler_fold.transform(X_fold_tr)
        X_fold_val_std = scaler_fold.transform(X_fold_val)

        # 2. PCA: aplicado somente quando o experimento exigir.
        if usar_pca:

            pca_fold = PCA(
                n_components=PCA_VARIANCE_TARGET
            ).fit(X_fold_tr_std)

            X_fold_tr_modelo = pca_fold.transform(X_fold_tr_std)
            X_fold_val_modelo = pca_fold.transform(X_fold_val_std)

            numero_componentes = pca_fold.components.shape[0]

            variancia_retida = float(
                np.sum(pca_fold.explained_variance_ratio)
            )

        else:

            X_fold_tr_modelo = X_fold_tr_std
            X_fold_val_modelo = X_fold_val_std

            numero_componentes = X_fold_tr_std.shape[1]
            variancia_retida = 1.0

        # 3. Ajuste e previsão do classificador da RBG.
        erro_ajuste = None

        try:

            modelo_fold = classe_modelo(
                labels=CLASSES_LDA_QDA
            )

            modelo_fold.fit(X_fold_tr_modelo, y_fold_tr)

            y_fold_pred = modelo_fold.predict(X_fold_val_modelo)

            metricas_fold = metricas_classificacao(
                y_fold_pred,
                y_fold_val
            )

        except (ValueError, np.linalg.LinAlgError) as erro:

            # Registramos eventuais problemas numéricos sem interromper
            # a execução dos outros folds e experimentos.
            metricas_fold = {
                "Acurácia": np.nan,
                "Precisão-macro": np.nan,
                "Recall-macro": np.nan,
                "F1-macro": np.nan,
                "F1-ponderado": np.nan,
            }

            erro_ajuste = str(erro)

        resultados.append({
            "Modelo": nome_modelo,
            "Fold": numero_fold,
            "Componentes": numero_componentes,
            "Variância retida": variancia_retida,
            **metricas_fold,
            "Erro de ajuste": erro_ajuste,
        })

    return pd.DataFrame(resultados)


A comparação inicial favoreceu o QDA, que apresentou F1-macro médio de 0,9235, contra 0,9197 do LDA, ambos sem PCA. Esse resultado sugere que a estimação de matrizes de covariância específicas por classe pode representar melhor as diferenças entre as variedades de feijão.

## Experimentos discriminante: LDA e QDA

Inicialmente, avaliamos os classificadores LDA e QDA utilizando exclusivamente as implementações disponíveis em `rbg-main/src/rbg/Classification`. Não substituímos os estimadores da biblioteca RBG por implementações de outras bibliotecas.

Nesta primeira etapa, são realizadas quatro configurações principais: LDA e QDA utilizando as 16 variáveis originais padronizadas e os mesmos classificadores após a aplicação do PCA com limiar de 90% de variância explicada.

Esses experimentos constituem a primeira parte da comparação. Posteriormente, a análise de sensibilidade investiga os limiares de 95% e 99%, completando as oito configurações avaliadas na validação cruzada.

### LDA sem PCA

In [10]:
resultados_lda_sem_pca = executar_experimento_cv(
    nome_modelo="LDA sem PCA",
    classe_modelo=LDA,
    usar_pca=False
)

print("Resultados do LDA sem PCA:")
display(resultados_lda_sem_pca.round(4))

Resultados do LDA sem PCA:


,Modelo,Fold,Componentes,Variância retida,Acurácia,Precisão-macro,Recall-macro,F1-macro,F1-ponderado,Erro de ajuste
0,LDA sem PCA,1,16,1.0,0.9088,0.9336,0.9217,0.9245,0.9103,None
1,LDA sem PCA,2,16,1.0,0.8999,0.9265,0.9110,0.9161,0.9019,None
2,LDA sem PCA,3,16,1.0,0.8952,0.9254,0.9093,0.9141,0.8972,None
3,LDA sem PCA,4,16,1.0,0.9090,0.9339,0.9200,0.9243,0.9106,None
4,LDA sem PCA,5,16,1.0,0.9034,0.9299,0.9147,0.9193,0.9049,None


### QDA sem PCA

In [11]:

resultados_qda_sem_pca = executar_experimento_cv(
    nome_modelo="QDA sem PCA",
    classe_modelo=QDA,
    usar_pca=False
)

print("Resultados do QDA sem PCA:")
display(resultados_qda_sem_pca.round(4))


Resultados do QDA sem PCA:


,Modelo,Fold,Componentes,Variância retida,Acurácia,Precisão-macro,Recall-macro,F1-macro,F1-ponderado,Erro de ajuste
0,QDA sem PCA,1,16,1.0,0.9111,0.9270,0.9286,0.9260,0.9116,None
1,QDA sem PCA,2,16,1.0,0.9137,0.9281,0.9274,0.9271,0.9143,None
2,QDA sem PCA,3,16,1.0,0.8989,0.9150,0.9165,0.9146,0.8991,None
3,QDA sem PCA,4,16,1.0,0.9173,0.9304,0.9318,0.9304,0.9176,None
4,QDA sem PCA,5,16,1.0,0.9048,0.9208,0.9197,0.9193,0.9054,None


### LDA com PCA

In [12]:
resultados_lda_com_pca = executar_experimento_cv(
    nome_modelo="LDA com PCA",
    classe_modelo=LDA,
    usar_pca=True
)

print("Resultados do LDA com PCA:")
display(resultados_lda_com_pca.round(4))


Resultados do LDA com PCA:


,Modelo,Fold,Componentes,Variância retida,Acurácia,Precisão-macro,Recall-macro,F1-macro,F1-ponderado,Erro de ajuste
0,LDA com PCA,1,4,0.9495,0.8834,0.8894,0.8842,0.8854,0.8834,None
1,LDA com PCA,2,4,0.9503,0.8810,0.8923,0.8817,0.8850,0.8807,None
2,LDA com PCA,3,4,0.9496,0.8661,0.8736,0.8676,0.8682,0.8649,None
3,LDA com PCA,4,4,0.9502,0.8808,0.8905,0.8823,0.8849,0.8802,None
4,LDA com PCA,5,4,0.9495,0.8826,0.8946,0.8867,0.8893,0.8825,None


### QDA com PCA

In [13]:
resultados_qda_com_pca = executar_experimento_cv(
    nome_modelo="QDA com PCA",
    classe_modelo=QDA,
    usar_pca=True
)

print("Resultados do QDA com PCA:")
display(resultados_qda_com_pca.round(4))

Resultados do QDA com PCA:


,Modelo,Fold,Componentes,Variância retida,Acurácia,Precisão-macro,Recall-macro,F1-macro,F1-ponderado,Erro de ajuste
0,QDA com PCA,1,4,0.9495,0.8899,0.8931,0.8907,0.8909,0.8899,None
1,QDA com PCA,2,4,0.9503,0.8898,0.8956,0.8919,0.8929,0.8896,None
2,QDA com PCA,3,4,0.9496,0.8758,0.8794,0.8795,0.8786,0.8748,None
3,QDA com PCA,4,4,0.9502,0.8905,0.8931,0.8951,0.8937,0.8899,None
4,QDA com PCA,5,4,0.9495,0.8835,0.8905,0.8886,0.8892,0.8833,None


## Análise de sensibilidade do limiar de PCA

Após os experimentos principais, investigamos como a escolha do limiar de variância explicada pelo PCA influencia o desempenho dos classificadores.

Como as configurações com limiar de 90% já foram avaliadas na etapa anterior, esta análise adicional considera os limiares de 95% e 99%, tanto para LDA quanto para QDA.

Para cada configuração, utilizamos as mesmas cinco partições da validação cruzada. Em cada fold, o `Standard` e o PCA são ajustados exclusivamente com as observações de treinamento, sendo posteriormente aplicados às observações de validação.

Dessa maneira, os resultados permitem comparar os diferentes níveis de redução dimensional sem utilizar o conjunto de teste externo e sem introduzir informações das observações de validação durante o ajuste do pré-processamento.

In [14]:
def avaliar_sensibilidade_pca(
    limiares: tuple[float, ...] = (0.95, 0.99)
) -> tuple[pd.DataFrame, pd.DataFrame]:

    resultados = []

    for limiar in limiares:

        for nome_modelo, classe_modelo in [
            ("LDA", LDA),
            ("QDA", QDA)
        ]:

            for fold, (idx_tr, idx_val) in enumerate(folds_cv, start=1):

                Xtr = X_train_raw[idx_tr]
                Xval = X_train_raw[idx_val]

                ytr = y_train[idx_tr]
                yval = y_train[idx_val]

                scaler = Standard().fit(Xtr)

                Xtr_std = scaler.transform(Xtr)
                Xval_std = scaler.transform(Xval)

                pca = PCA(
                    n_components=float(limiar)
                ).fit(Xtr_std)

                Xtr_pca = pca.transform(Xtr_std)
                Xval_pca = pca.transform(Xval_std)

                erro_ajuste = None

                try:
                    modelo = classe_modelo(
                        labels=CLASSES_LDA_QDA
                    )

                    modelo.fit(Xtr_pca, ytr)

                    pred = modelo.predict(Xval_pca)

                    metricas_fold = metricas_classificacao(
                        pred,
                        yval
                    )

                except (ValueError, np.linalg.LinAlgError) as erro:

                    metricas_fold = {
                        "Acurácia": np.nan,
                        "Precisão-macro": np.nan,
                        "Recall-macro": np.nan,
                        "F1-macro": np.nan,
                        "F1-ponderado": np.nan
                    }

                    erro_ajuste = str(erro)

                resultados.append({
                    "Limiar PCA": limiar,
                    "Modelo": nome_modelo,
                    "Fold": fold,
                    **metricas_fold,
                    "Componentes": pca.components.shape[0],
                    "Variância retida": float(
                        np.sum(pca.explained_variance_ratio)
                    ),
                    "Erro de ajuste": erro_ajuste
                })

    detalhe = pd.DataFrame(resultados)

    resumo = detalhe.groupby(
        ["Limiar PCA", "Modelo"]
    ).agg(
        F1_macro_medio=("F1-macro", "mean"),
        F1_macro_dp=("F1-macro", "std"),
        Acuracia_media=("Acurácia", "mean"),
        Precisao_macro_media=("Precisão-macro", "mean"),
        Recall_macro_medio=("Recall-macro", "mean"),
        F1_ponderado_medio=("F1-ponderado", "mean"),
        Folds_validos=("F1-macro", "count"),
        Componentes_medio=("Componentes", "mean"),
        Variancia_retida_media=("Variância retida", "mean")
    ).reset_index()

    return detalhe, resumo


df_sensibilidade_detalhe, df_sensibilidade_resumo = (
    avaliar_sensibilidade_pca()
)

print("Análise de sensibilidade do PCA:")
display(df_sensibilidade_resumo.round(4))


Análise de sensibilidade do PCA:


,Limiar PCA,Modelo,F1_macro_medio,F1_macro_dp,Acuracia_media,Precisao_macro_media,Recall_macro_medio,F1_ponderado_medio,Folds_validos,Componentes_medio,Variancia_retida_media
0,0.95,LDA,0.9054,0.0192,0.8953,0.9102,0.9032,0.8954,5,4.6,0.9665
1,0.95,QDA,0.9131,0.0190,0.9036,0.9129,0.9139,0.9035,5,4.6,0.9665
2,0.99,LDA,0.9159,0.0049,0.8992,0.9233,0.9134,0.9007,5,7.0,0.9960
3,0.99,QDA,0.9312,0.0078,0.9174,0.9309,0.9323,0.9175,5,7.0,0.9960


Os resultados mostram que a redução para 90% de variância prejudicou a classificação, enquanto a retenção de 99% apresentou melhora. O destaque foi o QDA com PCA de 99%, que alcançou F1-macro de 0,9312, superando o QDA sem PCA (0,9235). Portanto, o efeito do PCA depende do limiar escolhido, não sendo adequado descartá-lo como estratégia de pré-processamento.

In [15]:
# Consolidação dos oito cenários avaliados na validação cruzada.

def resumir_configuracao(
    nome: str,
    dados: pd.DataFrame
) -> dict:

    return {
        "Configuração": nome,
        "F1_macro_medio": dados["F1-macro"].mean(),
        "F1_macro_dp": dados["F1-macro"].std(),
        "Acuracia_media": dados["Acurácia"].mean(),
        "Precisao_macro_media": dados["Precisão-macro"].mean(),
        "Recall_macro_medio": dados["Recall-macro"].mean(),
        "F1_ponderado_medio": dados["F1-ponderado"].mean(),
        "Folds_validos": dados["F1-macro"].count(),
        "Componentes_medio": dados["Componentes"].mean(),
        "Variancia_retida_media": dados["Variância retida"].mean()
    }


linhas_resumo = []

# 1. Modelos sem PCA e modelos com PCA de 90%.
resultados_principais = [
    ("LDA sem PCA", resultados_lda_sem_pca),
    ("QDA sem PCA", resultados_qda_sem_pca),
    ("LDA — PCA 90%", resultados_lda_com_pca),
    ("QDA — PCA 90%", resultados_qda_com_pca)
]

for nome, dados in resultados_principais:
    linhas_resumo.append(
        resumir_configuracao(nome, dados)
    )


# Modelos com PCA de 95% e 99%.
for (limiar, nome_modelo), grupo in (
    df_sensibilidade_detalhe.groupby(
        ["Limiar PCA", "Modelo"]
    )
):

    nome_configuracao = (
        f"{nome_modelo} — PCA {limiar:.0%}"
    )

    linhas_resumo.append(
        resumir_configuracao(nome_configuracao, grupo)
    )


# Tabela consolidada.
resumo_cv_completo = (
    pd.DataFrame(linhas_resumo)
    .set_index("Configuração")
)

# Uma configuração só é elegível se tiver os cinco folds válidos.
resumo_cv_completo["Elegível"] = (
    resumo_cv_completo["Folds_validos"] == N_FOLDS_LDA_QDA
)

print("Comparação completa das oito configurações:")
display(
    resumo_cv_completo.sort_values(
        "F1_macro_medio",
        ascending=False
    ).round(4)
)


# Seleção do melhor modelo sem consultar o teste externo.
resumo_cv_validos = resumo_cv_completo[
    resumo_cv_completo["Elegível"]
]

if resumo_cv_validos.empty:
    raise RuntimeError(
        "Nenhuma configuração completou os cinco folds."
    )

# Maior F1-macro médio; em caso de empate, menor desvio-padrão.
ranking_cv = resumo_cv_validos.sort_values(
    by=["F1_macro_medio", "F1_macro_dp"],
    ascending=[False, True]
)

melhor_configuracao_cv = ranking_cv.index[0]

print("\nConfiguração selecionada pela validação cruzada:")
print(melhor_configuracao_cv)

display(
    ranking_cv.head(1).round(4)
)


Comparação completa das oito configurações:


,F1_macro_medio,F1_macro_dp,Acuracia_media,Precisao_macro_media,Recall_macro_medio,F1_ponderado_medio,Folds_validos,Componentes_medio,Variancia_retida_media,Elegível
Configuração,,,,,,,,,,
QDA — PCA 99%,0.9312,0.0078,0.9174,0.9309,0.9323,0.9175,5,7.0,0.9960,True
QDA sem PCA,0.9235,0.0064,0.9091,0.9242,0.9248,0.9096,5,16.0,1.0000,True
LDA sem PCA,0.9196,0.0047,0.9032,0.9298,0.9154,0.9050,5,16.0,1.0000,True
LDA — PCA 99%,0.9159,0.0049,0.8992,0.9233,0.9134,0.9007,5,7.0,0.9960,True
QDA — PCA 95%,0.9131,0.0190,0.9036,0.9129,0.9139,0.9035,5,4.6,0.9665,True
LDA — PCA 95%,0.9054,0.0192,0.8953,0.9102,0.9032,0.8954,5,4.6,0.9665,True
QDA — PCA 90%,0.8891,0.0061,0.8859,0.8903,0.8892,0.8855,5,4.0,0.9498,True
LDA — PCA 90%,0.8826,0.0082,0.8788,0.8881,0.8805,0.8783,5,4.0,0.9498,True



Configuração selecionada pela validação cruzada:
QDA — PCA 99%


,F1_macro_medio,F1_macro_dp,Acuracia_media,Precisao_macro_media,Recall_macro_medio,F1_ponderado_medio,Folds_validos,Componentes_medio,Variancia_retida_media,Elegível
Configuração,,,,,,,,,,
QDA — PCA 99%,0.9312,0.0078,0.9174,0.9309,0.9323,0.9175,5,7.0,0.996,True


## Ajuste final e avaliação no conjunto de teste

Após a comparação das oito configurações pela validação cruzada, selecionamos aquela que apresentou o maior F1-macro médio, considerando também o desvio-padrão entre os folds como critério de desempate.

A configuração selecionada foi o QDA com PCA de 99%. Nessa etapa, reajustamos o `Standard` e o PCA utilizando todas as observações disponíveis no conjunto de treinamento externo. Em seguida, aplicamos as transformações ao conjunto de teste, que permaneceu separado durante o processo de seleção dos modelos.

O classificador vencedor é então ajustado com todo o treinamento externo e utilizado para realizar as previsões no teste. Essa avaliação fornece uma estimativa de desempenho em observações que não participaram dos ajustes do classificador nem da seleção da configuração.

Também incluímos um baseline de classe majoritária, que atribui a todas as observações de teste a classe mais frequente no treinamento. Nesse caso, a classe majoritária é DERMASON. Essa referência permite verificar se o modelo selecionado apresenta capacidade de discriminação superior àquela obtida por uma estratégia que considera apenas a frequência das classes.

In [16]:
## Ajuste final e avaliação no conjunto de teste

# Associação entre os nomes da validação cruzada e os classificadores.
configuracoes = {
    "LDA sem PCA": (LDA, None),
    "QDA sem PCA": (QDA, None),
    "LDA — PCA 90%": (LDA, 0.90),
    "QDA — PCA 90%": (QDA, 0.90),
    "LDA — PCA 95%": (LDA, 0.95),
    "QDA — PCA 95%": (QDA, 0.95),
    "LDA — PCA 99%": (LDA, 0.99),
    "QDA — PCA 99%": (QDA, 0.99)
}

if melhor_configuracao_cv not in configuracoes:
    raise ValueError("Configuração selecionada não reconhecida.")

classe_modelo_final, limiar_pca_final = configuracoes[
    melhor_configuracao_cv
]


# 1. Ajuste da padronização somente no treino externo.
scaler_final = Standard().fit(X_train_raw)

X_train_final_std = scaler_final.transform(X_train_raw)
X_test_final_std = scaler_final.transform(X_test_raw)


# 2. Aplicação do PCA somente se a configuração selecionada exigir.
pca_final = None

if limiar_pca_final is not None:

    pca_final = PCA(
        n_components=limiar_pca_final
    ).fit(X_train_final_std)

    X_train_final = pca_final.transform(X_train_final_std)
    X_test_final = pca_final.transform(X_test_final_std)

    print(
        f"Variância retida: "
        f"{np.sum(pca_final.explained_variance_ratio) * 100:.2f}%"
    )

else:

    X_train_final = X_train_final_std
    X_test_final = X_test_final_std

    print("Modelo ajustado sem PCA.")


# 3. Ajuste do classificador vencedor e previsão no teste externo.
modelo_final = classe_modelo_final(
    labels=CLASSES_LDA_QDA
)

modelo_final.fit(X_train_final, y_train)

y_pred_final = modelo_final.predict(X_test_final)


# 4. Avaliação externa da configuração selecionada.
predicoes_lda_qda = {
    melhor_configuracao_cv: y_pred_final
}

resultados_teste_lda_qda = [{
    "Modelo": melhor_configuracao_cv,
    **metricas_classificacao(y_pred_final, y_test)
}]


# 5. Baseline de classe majoritária.
classe_majoritaria_treino = (
    pd.Series(y_train).value_counts().idxmax()
)

y_pred_baseline = np.full(
    y_test.shape,
    classe_majoritaria_treino,
    dtype=y_train.dtype
)

predicoes_lda_qda["Baseline majoritária"] = y_pred_baseline

resultados_teste_lda_qda.append({
    "Modelo": "Baseline majoritária",
    **metricas_classificacao(y_pred_baseline, y_test)
})


# 6. Resultados finais.
df_teste_lda_qda = (
    pd.DataFrame(resultados_teste_lda_qda)
    .set_index("Modelo")
)

print("Configuração selecionada:", melhor_configuracao_cv)
print("Classe majoritária do treino:", classe_majoritaria_treino)

print("\nAvaliação no conjunto de teste externo:")
display(df_teste_lda_qda.round(4))


Variância retida: 99.60%
Configuração selecionada: QDA — PCA 99%
Classe majoritária do treino: DERMASON

Avaliação no conjunto de teste externo:


,Acurácia,Precisão-macro,Recall-macro,F1-macro,F1-ponderado
Modelo,,,,,
QDA — PCA 99%,0.9233,0.9370,0.9354,0.9358,0.9237
Baseline majoritária,0.2618,0.0374,0.1429,0.0593,0.1086


## Diagnóstico por classe e matrizes de confusão

Além das métricas globais, examinamos o desempenho individual da configuração selecionada, o QDA com PCA de 99%, em cada uma das sete variedades de feijão.

Essa análise permite identificar classes que apresentam maior facilidade ou dificuldade de classificação, aspecto que pode não ser evidente apenas pela acurácia ou pelo F1-macro global.

São apresentados a precisão, o recall, o F1-score e o suporte de cada classe, acompanhados das respectivas matrizes de confusão. Nas matrizes, as linhas correspondem às classes verdadeiras e as colunas às classes previstas pelo classificador.

O baseline de classe majoritária também é analisado, permitindo observar como uma estratégia que prevê exclusivamente a classe DERMASON se comporta diante das diferentes variedades presentes no teste externo.

In [17]:
for nome_modelo, y_pred in predicoes_lda_qda.items():

    print(f"\n===== {nome_modelo}: métricas por classe =====")

    display(
        tabela_metricas_classe(y_pred, y_test).round(4)
    )

    print(
        f"Matriz de confusão — {nome_modelo} "
        "(linhas=real; colunas=previsto):"
    )

    display(
        matriz_confusao_rbg(
            y_test,
            y_pred,
            CLASSES_LDA_QDA
        )
    )


===== QDA — PCA 99%: métricas por classe =====


,Precisão,Recall,F1-score,Suporte
Classe,,,,
BARBUNYA,0.9409,0.9019,0.9210,265
BOMBAY,1.0000,1.0000,1.0000,105
CALI,0.9394,0.9509,0.9451,326
DERMASON,0.9409,0.8972,0.9185,710
HOROZ,0.9475,0.9704,0.9588,372
SEKER,0.9592,0.9261,0.9424,406
SIRA,0.8307,0.9015,0.8647,528


Matriz de confusão — QDA — PCA 99% (linhas=real; colunas=previsto):


Classe prevista,BARBUNYA,BOMBAY,CALI,DERMASON,HOROZ,SEKER,SIRA
Classe real,,,,,,,
BARBUNYA,239,0,16,0,2,3,5
BOMBAY,0,105,0,0,0,0,0
CALI,5,0,310,0,9,0,2
DERMASON,0,0,0,637,1,6,66
HOROZ,1,0,2,1,361,0,7
SEKER,6,0,0,7,0,376,17
SIRA,3,0,2,32,8,7,476



===== Baseline majoritária: métricas por classe =====


,Precisão,Recall,F1-score,Suporte
Classe,,,,
BARBUNYA,0.0000,0.0,0.000,265
BOMBAY,0.0000,0.0,0.000,105
CALI,0.0000,0.0,0.000,326
DERMASON,0.2618,1.0,0.415,710
HOROZ,0.0000,0.0,0.000,372
SEKER,0.0000,0.0,0.000,406
SIRA,0.0000,0.0,0.000,528


Matriz de confusão — Baseline majoritária (linhas=real; colunas=previsto):


Classe prevista,BARBUNYA,BOMBAY,CALI,DERMASON,HOROZ,SEKER,SIRA
Classe real,,,,,,,
BARBUNYA,0,0,0,265,0,0,0
BOMBAY,0,0,0,105,0,0,0
CALI,0,0,0,326,0,0,0
DERMASON,0,0,0,710,0,0,0
HOROZ,0,0,0,372,0,0,0
SEKER,0,0,0,406,0,0,0
SIRA,0,0,0,528,0,0,0


A análise por classe permite compreender melhor o comportamento do QDA com PCA de 99%, evidenciando diferenças de desempenho entre as variedades de feijão.

A classe BOMBAY apresentou o melhor resultado, com F1-score igual a 1,00, indicando classificação correta de todas as 105 observações presentes no teste. Também foram obtidos resultados elevados para HOROZ, com F1-score de 0,9588, CALI, com 0,9451, e SEKER, com 0,9424.

Por outro lado, SIRA apresentou o menor F1-score, igual a 0,8647, sendo a classe com maior dificuldade de classificação. DERMASON e BARBUNYA também apresentaram resultados relativamente inferiores aos das demais variedades, com F1-scores de 0,9185 e 0,9210, respectivamente.

A matriz de confusão permite identificar a principal origem desses erros. Foram observadas 66 amostras verdadeiramente pertencentes à classe DERMASON classificadas como SIRA, enquanto outras 32 amostras de SIRA foram classificadas como DERMASON. Portanto, a confusão entre essas duas variedades constitui a principal dificuldade observada no teste externo.

Também merece destaque a confusão entre BARBUNYA e CALI: 16 observações de BARBUNYA foram classificadas como CALI, enquanto cinco observações de CALI foram classificadas como BARBUNYA.

Esses resultados indicam que, embora o classificador apresente desempenho global elevado, determinadas variedades ainda possuem padrões de características que dificultam sua distinção. As confusões observadas sugerem possíveis sobreposições entre as representações dessas classes no espaço das variáveis utilizadas, embora a matriz, isoladamente, não permita estabelecer a causa dessas semelhanças.

## Análise dos resultados

Os experimentos realizados permitiram comparar o comportamento dos classificadores LDA e QDA em oito configurações distintas, considerando diferentes níveis de redução dimensional proporcionados pelo PCA. A comparação foi conduzida por meio de validação cruzada estratificada com cinco folds, utilizando as mesmas partições para todos os modelos.

Os resultados indicaram que o QDA apresentou desempenho superior ao LDA nas quatro condições de dimensionalidade consideradas. Entre todas as configurações avaliadas, o QDA com PCA de 99% obteve o maior F1-macro médio, alcançando 0,9312, com desvio-padrão de 0,0078. A segunda melhor configuração foi o QDA sem PCA, com F1-macro médio de 0,9235. O LDA sem redução dimensional apresentou resultado de 0,9196.

A análise também mostrou que o efeito do PCA depende do limiar escolhido. Para o LDA, nenhuma das configurações com PCA superou o resultado obtido sem redução dimensional. Para o QDA, os limiares de 90% e 95% produziram resultados inferiores ao modelo sem PCA, enquanto a configuração de 99% proporcionou o melhor resultado de toda a comparação. Portanto, não é possível afirmar que a redução dimensional melhora necessariamente a classificação: seus efeitos variam conforme o classificador e a quantidade de variância preservada.

A configuração selecionada pela validação cruzada foi então ajustada utilizando todo o conjunto de treinamento externo e avaliada em um conjunto de teste independente das etapas anteriores. O QDA com PCA de 99% alcançou acurácia de 0,9233 e F1-macro de 0,9358. Também foram obtidos precisão-macro de 0,9370, recall-macro de 0,9354 e F1 ponderado de 0,9237.

O F1-macro observado no teste externo foi ligeiramente superior à média obtida na validação cruzada, que correspondeu a 0,9312. Embora essa diferença não permita, isoladamente, estabelecer conclusões sobre a superioridade do modelo em outras amostras, os resultados indicam que o desempenho observado durante a validação cruzada foi mantido na avaliação externa.

A comparação com o baseline reforça a capacidade discriminativa do classificador selecionado. O baseline, que prevê exclusivamente a classe majoritária DERMASON, apresentou acurácia de 0,2618 e F1-macro de apenas 0,0593. Em contraste, o QDA com PCA de 99% alcançou F1-macro de 0,9358, demonstrando que seu desempenho não decorre apenas da predominância de determinada variedade no conjunto de dados.

A avaliação por classe acrescentou informações importantes à análise global. BOMBAY apresentou classificação perfeita no conjunto de teste, enquanto SIRA obteve o menor F1-score, igual a 0,8647. A matriz de confusão revelou que as principais dificuldades ocorreram na distinção entre DERMASON e SIRA, com erros nos dois sentidos, além de confusões entre BARBUNYA e CALI.

De maneira geral, os resultados mostram que o QDA foi particularmente adequado às configurações investigadas para o conjunto Dry Bean, especialmente quando associado ao PCA com limiar de 99%. A redução dimensional, nesse caso, permitiu representar as observações com um número menor de componentes, preservando aproximadamente 99,60% da variância e produzindo o melhor resultado de classificação entre os cenários avaliados.

Assim, considerando conjuntamente a validação cruzada, a avaliação no teste externo e o diagnóstico por classe, o QDA com PCA de 99% constitui a configuração de melhor desempenho neste estudo. Essa conclusão está restrita às partições, aos dados e às implementações utilizadas, mas evidencia a importância de avaliar experimentalmente diferentes combinações entre classificadores e técnicas de redução dimensional, em vez de presumir que uma configuração será sempre superior às demais.

In [18]:

## Síntese final dos experimentos

print("=" * 65)
print("SÍNTESE FINAL DOS EXPERIMENTOS LDA E QDA")
print("=" * 65)


# Resultados da configuração vencedora na validação cruzada.
resultado_cv_final = resumo_cv_completo.loc[
    [melhor_configuracao_cv],
    [
        "F1_macro_medio",
        "F1_macro_dp",
        "Acuracia_media",
        "Precisao_macro_media",
        "Recall_macro_medio",
        "F1_ponderado_medio"
    ]
].rename(columns={
    "F1_macro_medio": "F1-macro CV",
    "F1_macro_dp": "Desvio-padrão CV",
    "Acuracia_media": "Acurácia CV",
    "Precisao_macro_media": "Precisão-macro CV",
    "Recall_macro_medio": "Recall-macro CV",
    "F1_ponderado_medio": "F1-ponderado CV"
})


# Resultado externo da mesma configuração.
resultado_teste_final = df_teste_lda_qda.loc[
    [melhor_configuracao_cv],
    ["F1-macro", "Acurácia", "Precisão-macro", "Recall-macro"]
].rename(columns={
    "F1-macro": "F1-macro teste",
    "Acurácia": "Acurácia teste",
    "Precisão-macro": "Precisão-macro teste",
    "Recall-macro": "Recall-macro teste"
})


# Comparação sem incompatibilidade entre os índices.
df_comparacao_final = pd.concat(
    [resultado_cv_final, resultado_teste_final],
    axis=1
)

print("\nComparação entre validação cruzada e teste externo:")
display(df_comparacao_final.round(4))


print("\nConfiguração selecionada:")
print(melhor_configuracao_cv)

print(
    f"F1-macro médio na validação cruzada: "
    f"{resultado_cv_final.iloc[0]['F1-macro CV']:.4f}"
)

print(
    f"Desvio-padrão entre folds: "
    f"{resultado_cv_final.iloc[0]['Desvio-padrão CV']:.4f}"
)

print(
    f"F1-macro no teste externo: "
    f"{resultado_teste_final.iloc[0]['F1-macro teste']:.4f}"
)


print("\nBaseline de classe majoritária:")
display(
    df_teste_lda_qda.loc[
        ["Baseline majoritária"]
    ].round(4)
)


print("\nA seleção foi realizada exclusivamente pela validação cruzada.")
print("O conjunto de teste externo foi utilizado apenas para avaliação.")


SÍNTESE FINAL DOS EXPERIMENTOS LDA E QDA

Comparação entre validação cruzada e teste externo:


,F1-macro CV,Desvio-padrão CV,Acurácia CV,Precisão-macro CV,Recall-macro CV,F1-ponderado CV,F1-macro teste,Acurácia teste,Precisão-macro teste,Recall-macro teste
QDA — PCA 99%,0.9312,0.0078,0.9174,0.9309,0.9323,0.9175,0.9358,0.9233,0.937,0.9354



Configuração selecionada:
QDA — PCA 99%
F1-macro médio na validação cruzada: 0.9312
Desvio-padrão entre folds: 0.0078
F1-macro no teste externo: 0.9358

Baseline de classe majoritária:


,Acurácia,Precisão-macro,Recall-macro,F1-macro,F1-ponderado
Modelo,,,,,
Baseline majoritária,0.2618,0.0374,0.1429,0.0593,0.1086



A seleção foi realizada exclusivamente pela validação cruzada.
O conjunto de teste externo foi utilizado apenas para avaliação.
